# Impact of GDP per Capita on Vehicle Registrations using Time Series Tools

Reconstructed Python workflow based on the uploaded internship report. The report describes the analyses and results but does not contain the original source code, so this notebook reconstructs the corresponding Python workflow.

**Main analyses:** data preparation, Real GDP calculation, standardization, correlation, moving averages, ADF stationarity tests, linear detrending, Granger causality, VAR/AIC/BIC modelling, forecasting, 2-wheeler vs 4-wheeler analysis, YoY growth, income elasticity, and related accident/ownership plots.

In [ ]:
# ============================================================
# INTERNSHIP PROJECT:
# Impact of GDP per Capita on Vehicle Registrations
# using Time Series Tools
#
# Reconstructed Python workflow based on the uploaded report.
# NOTE:
# The PDF describes the analyses/results but does NOT contain
# the original source code. Therefore, this is a faithful
# reconstruction of the code required to reproduce the analyses,
# using the variables and methods described in the report.


In [ ]:
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL / IMPORT LIBRARIES
# ------------------------------------------------------------
# If needed, run these in Jupyter:
# !pip install pandas numpy matplotlib seaborn scipy scikit-learn statsmodels openpyxl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
import statsmodels.api as sm

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True


# ------------------------------------------------------------
# 2. LOAD THE MAIN DATA
# ------------------------------------------------------------
# Example Excel structure:
#
# Year | Vehicle Registrations | 2-Wheeler | 4-Wheeler |
#      | Nominal GDP | GDP per Capita | GDP Deflator
#
# Change the file name and sheet name according to your file.

df = pd.read_excel("vehicle_gdp_data.xlsx")

print(df.head())
print(df.info())
print(df.describe())


# ------------------------------------------------------------
# 3. CLEAN / ORGANISE DATA
# ------------------------------------------------------------

# Remove completely empty rows
df = df.dropna(how="all")

# Convert Year to integer
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# Convert numerical columns to numeric
numeric_columns = [
    "Vehicle Registrations",
    "2-Wheeler",
    "4-Wheeler",
    "Nominal GDP",
    "GDP per Capita",
    "GDP Deflator"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Sort by year
df = df.sort_values("Year").reset_index(drop=True)

print(df.head())
print(df.tail())


# ------------------------------------------------------------
# 4. CALCULATE REAL GDP
# ------------------------------------------------------------
# Report formula:
#
# Real GDP = (Nominal GDP / GDP Deflator) * 100

df["Real GDP"] = (
    df["Nominal GDP"] / df["GDP Deflator"]
) * 100

print(df[["Year", "Nominal GDP", "GDP Deflator", "Real GDP"]].head())


# ------------------------------------------------------------
# 5. CHECK MISSING VALUES
# ------------------------------------------------------------

print(df.isnull().sum())

# If appropriate for the dataset:
df = df.dropna(subset=[
    "Year",
    "Vehicle Registrations",
    "Nominal GDP",
    "Real GDP",
    "GDP per Capita"
]).reset_index(drop=True)


# ------------------------------------------------------------
# 6. INITIAL PLOTS OF THE FOUR MAIN VARIABLES
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

plt.plot(
    df["Year"],
    df["Nominal GDP"],
    marker="o",
    label="Nominal GDP"
)

plt.plot(
    df["Year"],
    df["Real GDP"],
    marker="o",
    label="Real GDP"
)

plt.plot(
    df["Year"],
    df["GDP per Capita"],
    marker="o",
    label="GDP per Capita"
)

plt.plot(
    df["Year"],
    df["Vehicle Registrations"],
    marker="o",
    label="Vehicle Registrations"
)

plt.xlabel("Year")
plt.ylabel("Value")
plt.title("GDP and Vehicle Registrations over Time")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 7. Z-SCORE STANDARDIZATION
# ------------------------------------------------------------
# Formula:
# z = (x - mean) / standard deviation
#
# Standardization is required because GDP and vehicle
# registrations have very different units/scales.

variables = [
    "Nominal GDP",
    "Real GDP",
    "GDP per Capita",
    "Vehicle Registrations"
]

scaler = StandardScaler()

standardized = scaler.fit_transform(df[variables])

standardized_df = pd.DataFrame(
    standardized,
    columns=[
        "NGs",
        "RGs",
        "GpCs",
        "VR"
    ]
)

standardized_df["Year"] = df["Year"].values

print(standardized_df.head())


# ------------------------------------------------------------
# 8. PLOT STANDARDIZED VARIABLES
# ------------------------------------------------------------

plt.figure(figsize=(14, 7))

plt.plot(
    standardized_df["Year"],
    standardized_df["NGs"],
    marker="o",
    label="Standardized Nominal GDP"
)

plt.plot(
    standardized_df["Year"],
    standardized_df["RGs"],
    marker="o",
    label="Standardized Real GDP"
)

plt.plot(
    standardized_df["Year"],
    standardized_df["GpCs"],
    marker="o",
    label="Standardized GDP per Capita"
)

plt.plot(
    standardized_df["Year"],
    standardized_df["VR"],
    marker="x",
    label="Standardized Vehicle Registrations"
)

plt.xlabel("Years")
plt.ylabel("Z-score")
plt.title("Standardized GDP and Vehicle Registrations")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 9. CORRELATION MATRIX
# ------------------------------------------------------------

corr = standardized_df[
    ["NGs", "RGs", "GpCs", "VR"]
].corr()

print(corr)


# ------------------------------------------------------------
# 10. CORRELATION HEATMAP
# ------------------------------------------------------------

plt.figure(figsize=(8, 6))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 11. 5-YEAR MOVING AVERAGE
# ------------------------------------------------------------

for col in ["NGs", "RGs", "GpCs", "VR"]:
    standardized_df[f"{col}_MA5"] = (
        standardized_df[col]
        .rolling(window=5)
        .mean()
    )


# ------------------------------------------------------------
# 12. PLOT 5-YEAR MOVING AVERAGES
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

series_info = [
    ("NGs", "Nominal GDP"),
    ("RGs", "Real GDP"),
    ("GpCs", "GDP per Capita"),
    ("VR", "Vehicle Registrations")
]

for ax, (col, title) in zip(axes.flatten(), series_info):

    ax.plot(
        standardized_df["Year"],
        standardized_df[col],
        label="Original"
    )

    ax.plot(
        standardized_df["Year"],
        standardized_df[f"{col}_MA5"],
        label="Trend (5-year MA)",
        linewidth=2
    )

    ax.set_title(f"MA Trend in {title}")
    ax.set_xlabel("Year")
    ax.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 13. ADF TEST FUNCTION
# ------------------------------------------------------------

def adf_test(series, name):
    """
    Augmented Dickey-Fuller test.
    H0: Unit root exists -> non-stationary
    H1: No unit root -> stationary
    """

    series = pd.Series(series).dropna()

    result = adfuller(series, autolag="AIC")

    print("\n", "=" * 50)
    print("ADF TEST:", name)
    print("=" * 50)

    print("ADF Statistic:", result[0])
    print("p-value:", result[1])
    print("Used Lags:", result[2])
    print("Number of Observations:", result[3])

    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value}")

    if result[1] < 0.05:
        print("Conclusion: Stationary (reject H0)")
    else:
        print("Conclusion: Non-stationary (fail to reject H0)")

    return result


# ------------------------------------------------------------
# 14. ADF TEST BEFORE DE-TRENDING
# ------------------------------------------------------------

adf_results_before = {}

for col in ["NGs", "RGs", "GpCs", "VR"]:
    adf_results_before[col] = adf_test(
        standardized_df[col],
        col
    )


# ------------------------------------------------------------
# 15. LINEAR TREND FITTING / DE-TRENDING
# ------------------------------------------------------------
# For each standardized series:
#     y = a + b * Year
#
# Residual = Actual - Fitted Trend

def detrend_series(year, values):

    X = np.asarray(year).reshape(-1, 1)
    y = np.asarray(values)

    model = LinearRegression()
    model.fit(X, y)

    fitted = model.predict(X)
    residuals = y - fitted

    slope = model.coef_[0]
    intercept = model.intercept_

    return model, fitted, residuals, slope, intercept


residuals = pd.DataFrame()
residuals["Year"] = standardized_df["Year"]

trend_models = {}

for col in ["NGs", "RGs", "GpCs", "VR"]:

    model, fitted, resid, slope, intercept = detrend_series(
        standardized_df["Year"],
        standardized_df[col]
    )

    trend_models[col] = model
    standardized_df[f"{col}_Trend"] = fitted
    residuals[col] = resid

    print(
        f"{col}: y = {slope:.4f} * Year + {intercept:.4f}"
    )


# ------------------------------------------------------------
# 16. PLOT ORIGINAL SERIES + LINEAR TREND
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), ["NGs", "RGs", "GpCs", "VR"]):

    ax.plot(
        standardized_df["Year"],
        standardized_df[col],
        label=col
    )

    ax.plot(
        standardized_df["Year"],
        standardized_df[f"{col}_Trend"],
        label="Trend line"
    )

    slope = trend_models[col].coef_[0]
    intercept = trend_models[col].intercept_

    ax.set_title(f"Fitting Linear Regression {col}")

    ax.text(
        0.03,
        0.93,
        f"Trend equation: y = {slope:.3f}x + {intercept:.2f}",
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="top"
    )

    ax.legend()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 17. PLOT DE-TRENDED RESIDUALS
# ------------------------------------------------------------

plt.figure(figsize=(14, 7))

plt.plot(
    residuals["Year"],
    residuals["VR"],
    label="Residual of Vehicle Registrations"
)

plt.plot(
    residuals["Year"],
    residuals["NGs"],
    label="Residual of Nominal GDP"
)

plt.plot(
    residuals["Year"],
    residuals["RGs"],
    label="Residual of Real GDP"
)

plt.plot(
    residuals["Year"],
    residuals["GpCs"],
    label="Residual of GDP per Capita"
)

plt.xlabel("Year")
plt.ylabel("Residual")
plt.title("De-trended Residuals")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 18. ADF TEST AFTER DE-TRENDING
# ------------------------------------------------------------

adf_results_after = {}

for col in ["NGs", "RGs", "GpCs", "VR"]:

    adf_results_after[col] = adf_test(
        residuals[col],
        f"Residual {col}"
    )


# ------------------------------------------------------------
# 19. CREATE ADF P-VALUE TABLE
# ------------------------------------------------------------

before_table = pd.DataFrame({
    "Series": list(adf_results_before.keys()),
    "p_value": [
        result[1]
        for result in adf_results_before.values()
    ]
})

after_table = pd.DataFrame({
    "Series": list(adf_results_after.keys()),
    "p_value": [
        result[1]
        for result in adf_results_after.values()
    ]
})

print("\nADF p-values BEFORE de-trending")
print(before_table)

print("\nADF p-values AFTER de-trending")
print(after_table)


# ------------------------------------------------------------
# 20. GRANGER CAUSALITY:
# GDP PER CAPITA -> VEHICLE REGISTRATIONS
# ------------------------------------------------------------
# Report tests lags 1 through 7.
#
# The two variables used here are the de-trended residuals.

granger_data = residuals[
    ["VR", "GpCs"]
].dropna()

granger_pvalues = []

for lag in range(1, 8):

    result = grangercausalitytests(
        granger_data[["VR", "GpCs"]],
        maxlag=lag,
        verbose=False
    )

    # F-test p-value for the requested lag
    p_value = result[lag][0]["ssr_ftest"][1]

    granger_pvalues.append({
        "Lag": lag,
        "p_value": p_value
    })

granger_table = pd.DataFrame(granger_pvalues)

print("\nGranger Causality: GDP per Capita -> Vehicle Registrations")
print(granger_table)


# ------------------------------------------------------------
# 21. PLOT GRANGER P-VALUES
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

plt.plot(
    granger_table["Lag"],
    granger_table["p_value"],
    marker="o"
)

plt.axhline(
    0.05,
    linestyle="--",
    label="5% significance level"
)

plt.xlabel("Lag")
plt.ylabel("p-value")
plt.title("Granger Causality p-values")
plt.xticks(range(1, 8))
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 22. CHECK BOTH DIRECTIONS OF GRANGER CAUSALITY
# ------------------------------------------------------------
# Direction 1:
# GDP per capita -> Vehicle Registrations
#
# Direction 2:
# Vehicle Registrations -> GDP per capita

print("\nGDP per Capita -> Vehicle Registrations")

grangercausalitytests(
    granger_data[["VR", "GpCs"]],
    maxlag=7,
    verbose=True
)

print("\nVehicle Registrations -> GDP per Capita")

grangercausalitytests(
    granger_data[["GpCs", "VR"]],
    maxlag=7,
    verbose=True
)


# ------------------------------------------------------------
# 23. PREPARE DATA FOR VAR MODEL
# ------------------------------------------------------------
# The report forecasts residual vehicle registrations.
#
# For a simple two-variable VAR:
#     residual Vehicle Registrations
#     residual GDP per Capita
#
# You can add other stationary residuals if required.

var_data = residuals[
    ["VR", "GpCs"]
].dropna()

print(var_data.head())


# ------------------------------------------------------------
# 24. SELECT VAR LAG USING AIC AND BIC
# ------------------------------------------------------------

var_model = VAR(var_data)

lag_selection = var_model.select_order(maxlags=7)

print(lag_selection.summary())

print("\nAIC selected lag:", lag_selection.aic)
print("BIC selected lag:", lag_selection.bic)


# ------------------------------------------------------------
# 25. CREATE AIC/BIC TABLE
# ------------------------------------------------------------

aic_values = lag_selection.ics["aic"]
bic_values = lag_selection.ics["bic"]

lag_table = pd.DataFrame({
    "Lag": range(len(aic_values)),
    "AIC": aic_values,
    "BIC": bic_values
})

print(lag_table)


# ------------------------------------------------------------
# 26. FIT VAR MODEL AT LAG 2
# ------------------------------------------------------------
# The report selected lag 2 after comparing AIC and BIC.

var_results = var_model.fit(2)

print(var_results.summary())


# ------------------------------------------------------------
# 27. FORECAST NEXT 10 YEARS
# ------------------------------------------------------------

lag_order = var_results.k_ar

last_values = var_data.values[-lag_order:]

forecast = var_results.forecast(
    y=last_values,
    steps=10
)

forecast_df = pd.DataFrame(
    forecast,
    columns=var_data.columns
)

future_years = np.arange(
    df["Year"].max() + 1,
    df["Year"].max() + 11
)

forecast_df["Year"] = future_years

print(forecast_df)


# ------------------------------------------------------------
# 28. PLOT FORECASTED RESIDUAL VEHICLE REGISTRATIONS
# ------------------------------------------------------------

plt.figure(figsize=(12, 6))

plt.plot(
    forecast_df["Year"],
    forecast_df["VR"],
    marker="o",
    label="Predicted residuals of Vehicle Registrations"
)

plt.xlabel("Year")
plt.ylabel("Predicted Residual")
plt.title("10-Year Forecast of Vehicle Registration Residuals")
plt.legend()
plt.tight_layout()
plt.show()




In [ ]:
# ============================================================
# 29. TWO-WHEELER AND FOUR-WHEELER ANALYSIS


In [ ]:
# ============================================================

# Required columns:
# "2-Wheeler", "4-Wheeler", "GDP per Capita"

tw = df[
    ["Year", "2-Wheeler", "4-Wheeler", "GDP per Capita"]
].copy()


# ------------------------------------------------------------
# 30. PRE-STANDARDIZATION PLOT
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

plt.plot(
    tw["Year"],
    tw["2-Wheeler"],
    marker="o",
    label="2-wheeler registrations"
)

plt.plot(
    tw["Year"],
    tw["4-Wheeler"],
    marker="o",
    label="4-wheeler registrations"
)

plt.plot(
    tw["Year"],
    tw["GDP per Capita"],
    marker="o",
    label="GDP per capita"
)

plt.xlabel("Year")
plt.ylabel("Value")
plt.title("2-Wheeler, 4-Wheeler and GDP per Capita")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 31. STANDARDIZE 2-WHEELER, 4-WHEELER AND GDP PER CAPITA
# ------------------------------------------------------------

tw_scaler = StandardScaler()

tw_standardized = tw_scaler.fit_transform(
    tw[
        ["2-Wheeler", "4-Wheeler", "GDP per Capita"]
    ]
)

tw_std = pd.DataFrame(
    tw_standardized,
    columns=[
        "2W_std",
        "4W_std",
        "GpC_std"
    ]
)

tw_std["Year"] = tw["Year"].values


# ------------------------------------------------------------
# 32. POST-STANDARDIZATION PLOT
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

plt.plot(
    tw_std["Year"],
    tw_std["2W_std"],
    marker="o",
    label="Standardized 2-wheeler registrations"
)

plt.plot(
    tw_std["Year"],
    tw_std["4W_std"],
    marker="o",
    label="Standardized 4-wheeler registrations"
)

plt.plot(
    tw_std["Year"],
    tw_std["GpC_std"],
    marker="o",
    label="Standardized GDP per capita"
)

plt.xlabel("Year")
plt.ylabel("Z-score")
plt.title("Standardized 2-Wheeler, 4-Wheeler and GDP per Capita")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 33. YEAR-ON-YEAR GROWTH
# ------------------------------------------------------------

tw["2W_YoY"] = tw["2-Wheeler"].pct_change() * 100
tw["4W_YoY"] = tw["4-Wheeler"].pct_change() * 100

print(
    tw[
        ["Year", "2W_YoY", "4W_YoY"]
    ]
)


# ------------------------------------------------------------
# 34. PLOT YoY GROWTH
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

plt.plot(
    tw["Year"],
    tw["2W_YoY"],
    marker="o",
    label="Growth Rate of 2 Wheelers"
)

plt.plot(
    tw["Year"],
    tw["4W_YoY"],
    marker="o",
    label="Growth Rate of 4 Wheelers"
)

plt.axhline(0, linestyle="--")

plt.xlabel("Year")
plt.ylabel("YoY Growth (%)")
plt.title("YoY Growth Rate of 2-Wheelers vs 4-Wheelers")
plt.legend()
plt.tight_layout()
plt.show()




In [ ]:
# ============================================================
# 35. INCOME ELASTICITY USING LOG-LOG REGRESSION


In [ ]:
# ============================================================
# Model:
#
# log(Vehicle Demand) =
#     alpha + beta * log(GDP per Capita) + error
#
# beta = income elasticity


tw["log_2W"] = np.log(tw["2-Wheeler"])
tw["log_4W"] = np.log(tw["4-Wheeler"])
tw["log_GpC"] = np.log(tw["GDP per Capita"])


# ------------------------------------------------------------
# 36. LOG-LOG REGRESSION FOR 2-WHEELERS
# ------------------------------------------------------------

X_2W = sm.add_constant(tw["log_GpC"])

model_2W = sm.OLS(
    tw["log_2W"],
    X_2W
).fit()

print("\n2-Wheeler Income Elasticity")
print(model_2W.summary())

elasticity_2W = model_2W.params["log_GpC"]

print(
    "2-Wheeler income elasticity:",
    elasticity_2W
)


# ------------------------------------------------------------
# 37. LOG-LOG REGRESSION FOR 4-WHEELERS
# ------------------------------------------------------------

X_4W = sm.add_constant(tw["log_GpC"])

model_4W = sm.OLS(
    tw["log_4W"],
    X_4W
).fit()

print("\n4-Wheeler Income Elasticity")
print(model_4W.summary())

elasticity_4W = model_4W.params["log_GpC"]

print(
    "4-Wheeler income elasticity:",
    elasticity_4W
)


# ------------------------------------------------------------
# 38. OPTIONAL: SCIKIT-LEARN VERSION OF LOG-LOG REGRESSION
# ------------------------------------------------------------

X = tw[["log_GpC"]]

y_2W = tw["log_2W"]
y_4W = tw["log_4W"]

lr_2W = LinearRegression()
lr_2W.fit(X, y_2W)

lr_4W = LinearRegression()
lr_4W.fit(X, y_4W)

print("2W elasticity:", lr_2W.coef_[0])
print("4W elasticity:", lr_4W.coef_[0])




In [ ]:
# ============================================================
# 39. SEPARATE TIME-SERIES ANALYSIS FOR 2-WHEELERS


In [ ]:
# ============================================================

tw_analysis = tw_std[
    ["Year", "2W_std", "GpC_std"]
].copy()

model_2w_trend, trend_2w, resid_2w, _, _ = detrend_series(
    tw_analysis["Year"],
    tw_analysis["2W_std"]
)

model_gpc_trend, trend_gpc, resid_gpc, _, _ = detrend_series(
    tw_analysis["Year"],
    tw_analysis["GpC_std"]
)

tw_residuals = pd.DataFrame({
    "Year": tw_analysis["Year"],
    "2W": resid_2w,
    "GpC": resid_gpc
})


# ------------------------------------------------------------
# 40. ADF FOR 2-WHEELER AND GDP PER CAPITA RESIDUALS
# ------------------------------------------------------------

adf_2w = adf_test(
    tw_residuals["2W"],
    "2-Wheeler residual"
)

adf_gpc_2w = adf_test(
    tw_residuals["GpC"],
    "GDP per Capita residual"
)


# ------------------------------------------------------------
# 41. GRANGER TEST: GDP PER CAPITA -> 2-WHEELERS
# ------------------------------------------------------------

data_2w_granger = tw_residuals[
    ["2W", "GpC"]
].dropna()

print("\nGDP per Capita -> 2-Wheelers")

granger_2w = []

for lag in range(1, 8):

    result = grangercausalitytests(
        data_2w_granger[["2W", "GpC"]],
        maxlag=lag,
        verbose=False
    )

    p = result[lag][0]["ssr_ftest"][1]

    granger_2w.append({
        "Lag": lag,
        "p_value": p
    })

granger_2w = pd.DataFrame(granger_2w)

print(granger_2w)




In [ ]:
# ============================================================
# 42. SEPARATE TIME-SERIES ANALYSIS FOR 4-WHEELERS


In [ ]:
# ============================================================

model_4w_trend, trend_4w, resid_4w, _, _ = detrend_series(
    tw_analysis["Year"],
    tw_analysis["4W_std"]
)

tw_residuals["4W"] = resid_4w


# ------------------------------------------------------------
# 43. ADF FOR 4-WHEELER RESIDUAL
# ------------------------------------------------------------

adf_4w = adf_test(
    tw_residuals["4W"],
    "4-Wheeler residual"
)


# ------------------------------------------------------------
# 44. GRANGER TEST: GDP PER CAPITA -> 4-WHEELERS
# ------------------------------------------------------------

data_4w_granger = tw_residuals[
    ["4W", "GpC"]
].dropna()

print("\nGDP per Capita -> 4-Wheelers")

granger_4w = []

for lag in range(1, 8):

    result = grangercausalitytests(
        data_4w_granger[["4W", "GpC"]],
        maxlag=lag,
        verbose=False
    )

    p = result[lag][0]["ssr_ftest"][1]

    granger_4w.append({
        "Lag": lag,
        "p_value": p
    })

granger_4w = pd.DataFrame(granger_4w)

print(granger_4w)


# ------------------------------------------------------------
# 45. VAR FOR 2-WHEELER + 4-WHEELER + GDP PER CAPITA
# ------------------------------------------------------------
# This corresponds to the report's separate VAR analysis.
# A two-variable version can also be used for each category.

var_vehicle_data = tw_residuals[
    ["2W", "4W", "GpC"]
].dropna()

var_vehicle = VAR(var_vehicle_data)

vehicle_lag_selection = var_vehicle.select_order(
    maxlags=7
)

print(vehicle_lag_selection.summary())

# Report uses lag 2
vehicle_var_results = var_vehicle.fit(2)

print(vehicle_var_results.summary())


# ------------------------------------------------------------
# 46. FORECAST 2-WHEELER AND 4-WHEELER RESIDUALS
# ------------------------------------------------------------

lag_order = vehicle_var_results.k_ar

last_values = var_vehicle_data.values[-lag_order:]

vehicle_forecast = vehicle_var_results.forecast(
    y=last_values,
    steps=10
)

future_vehicle_years = np.arange(
    tw["Year"].max() + 1,
    tw["Year"].max() + 11
)

vehicle_forecast_df = pd.DataFrame(
    vehicle_forecast,
    columns=var_vehicle_data.columns
)

vehicle_forecast_df["Year"] = future_vehicle_years

print(vehicle_forecast_df)


# ------------------------------------------------------------
# 47. PLOT 2-WHEELER AND 4-WHEELER FORECASTED RESIDUALS
# ------------------------------------------------------------

plt.figure(figsize=(12, 7))

plt.plot(
    vehicle_forecast_df["Year"],
    vehicle_forecast_df["2W"],
    marker="o",
    label="Predicted residuals of 2-Wheeler registrations"
)

plt.plot(
    vehicle_forecast_df["Year"],
    vehicle_forecast_df["4W"],
    marker="o",
    label="Predicted residuals of 4-Wheeler registrations"
)

plt.xlabel("Year")
plt.ylabel("Predicted Residual")
plt.title("Forecasted Residuals of 2-Wheelers and 4-Wheelers")
plt.legend()
plt.tight_layout()
plt.show()




In [ ]:
# ============================================================
# 48. OPTIONAL ACCIDENT DATA ANALYSIS


In [ ]:
# ============================================================
# The report also contains accident/ownership charts.
# The exact source files and exact column names are not included
# in the PDF, so these sections require the corresponding
# accident/ownership datasets.
#
# Example expected accident data:
#
# Vehicle Type | Percentage
#
# Example expected rural/urban data:
#
# Year | Rural | Urban
#
# ------------------------------------------------------------

# accident_df = pd.read_excel("road_accident_data.xlsx")

# Example horizontal bar chart:
#
# accident_df = accident_df.sort_values("Percentage")
#
# plt.figure(figsize=(10, 7))
# plt.barh(
#     accident_df["Vehicle Type"],
#     accident_df["Percentage"]
# )
# plt.xlabel("Percentage")
# plt.ylabel("Vehicle Type")
# plt.title("Road Accident Deaths by Vehicle Type")
# plt.tight_layout()
# plt.show()


# ------------------------------------------------------------
# 49. RURAL VS URBAN ROAD ACCIDENT FATALITIES
# ------------------------------------------------------------

# accident_ru = pd.read_excel("rural_urban_accidents.xlsx")
#
# plt.figure(figsize=(12, 7))
#
# plt.plot(
#     accident_ru["Year"],
#     accident_ru["Rural"],
#     marker="o",
#     label="Rural"
# )
#
# plt.plot(
#     accident_ru["Year"],
#     accident_ru["Urban"],
#     marker="o",
#     label="Urban"
# )
#
# # Linear trend lines
# rural_model = np.polyfit(
#     accident_ru["Year"],
#     accident_ru["Rural"],
#     1
# )
#
# urban_model = np.polyfit(
#     accident_ru["Year"],
#     accident_ru["Urban"],
#     1
# )
#
# rural_trend = np.polyval(
#     rural_model,
#     accident_ru["Year"]
# )
#
# urban_trend = np.polyval(
#     urban_model,
#     accident_ru["Year"]
# )
#
# plt.plot(
#     accident_ru["Year"],
#     rural_trend,
#     linestyle="--",
#     label="Linear (Rural)"
# )
#
# plt.plot(
#     accident_ru["Year"],
#     urban_trend,
#     linestyle="--",
#     label="Linear (Urban)"
# )
#
# plt.xlabel("Year")
# plt.ylabel("Road Accident Fatalities")
# plt.title("Road Accident Fatalities - Rural vs Urban")
# plt.legend()
# plt.tight_layout()
# plt.show()




In [ ]:
# ============================================================
# 50. RURAL / URBAN VEHICLE OWNERSHIP


In [ ]:
# ============================================================
# The report shows separate graphs for 2-wheelers and 4-wheelers.
# Exact source data is not present in the PDF.
#
# Expected structure:
#
# Year | Rural | Urban
#
# ------------------------------------------------------------

# ownership_2w = pd.read_excel("2w_ownership.xlsx")
#
# plt.figure(figsize=(10, 6))
#
# plt.plot(
#     ownership_2w["Year"],
#     ownership_2w["Rural"],
#     marker="o",
#     label="Share Of Households in Rural (%)"
# )
#
# plt.plot(
#     ownership_2w["Year"],
#     ownership_2w["Urban"],
#     marker="o",
#     label="Share Of Households in Urban (%)"
# )
#
# plt.title("Growth rate of 2-wheeler ownership (base 1983)")
# plt.xlabel("Year")
# plt.ylabel("Percentage")
# plt.legend()
# plt.tight_layout()
# plt.show()




In [ ]:
# ============================================================
# 51. SAVE IMPORTANT OUTPUT TABLES


In [ ]:
# ============================================================

standardized_df.to_csv(
    "standardized_time_series.csv",
    index=False
)

residuals.to_csv(
    "detrended_residuals.csv",
    index=False
)

before_table.to_csv(
    "ADF_before_detrending.csv",
    index=False
)

after_table.to_csv(
    "ADF_after_detrending.csv",
    index=False
)

granger_table.to_csv(
    "granger_causality.csv",
    index=False
)

lag_table.to_csv(
    "VAR_lag_selection.csv",
    index=False
)

forecast_df.to_csv(
    "vehicle_registration_forecast.csv",
    index=False
)

tw.to_csv(
    "two_four_wheeler_analysis.csv",
    index=False
)

granger_2w.to_csv(
    "granger_2w.csv",
    index=False
)

granger_4w.to_csv(
    "granger_4w.csv",
    index=False
)

vehicle_forecast_df.to_csv(
    "two_four_wheeler_forecast.csv",
    index=False
)




In [ ]:
# ============================================================
# 52. FINAL SUMMARY OF MAIN RESULTS


In [ ]:
# ============================================================

print("\n================ FINAL SUMMARY ================")

print("2-Wheeler income elasticity:", elasticity_2W)
print("4-Wheeler income elasticity:", elasticity_4W)

print("\nGranger causality results:")
print(granger_table)

print("\nVAR lag-selection table:")
print(lag_table)

print("\n10-year Vehicle Registration residual forecast:")
print(forecast_df)

print("\n10-year 2W/4W residual forecast:")
print(vehicle_forecast_df)

print("\nAnalysis completed.")
